# CAR

O Serviço Florestal Brasileiro mantem um GeoServer sob um certificado TLS 1.2 (ultrapassado) e isso dificulta o acesso usando a biblioteca [`OWSLib`](https://owslib.readthedocs.io/en/latest/), como faço usualmente.

Cheguei a abri a _issue_ [Cipher / SSL Error #1008](https://github.com/geopython/OWSLib/issues/1008), buscando auxílio e os caminhos indicados me fizeram fazer o acesso diretamente por meio de requisição _web_, abstraindo o GeoServer.


In [ ]:
import json
import pprint
import tempfile

import ee
import geemap
import geopandas as gpd
import pandas as pd

import open_geodata as geo

In [ ]:
import requests
from urllib3.util import create_urllib3_context
from urllib.parse import parse_qsl, urlencode, urlsplit, urlunsplit

<br>

Vetores


In [ ]:
# Obtem, seleciona
gdf = geo.load_dataset(db='sp', name='geo.sp_250k_wgs84')

# Seleciona
mask = (
    (gdf['municipio_nome'] == 'Piracicaba')
    | (gdf['municipio_nome'] == 'Americana')
    | (gdf['municipio_nome'] == 'Limeira')
)
gdf = gdf[mask]

# Passa para a mesma coordenada do raster
# gdf['geometry'] = gdf['geometry'].to_crs(4326)
gdf['geometry'] = gdf['geometry'].to_crs(4674)

# Reseta Index
gdf = gdf.reset_index(drop=True)

gdf.info()
gdf.head()

<br>

Criamos uma pasta temporária, onde os arquivos serão baixados.


In [ ]:
# Crio pasta temporária
temp_path = Path(tempfile.gettempdir()) / 'open_geodata' / 'sfb'
temp_path.mkdir(exist_ok=True, parents=True)
temp_path

<br>

Definimos qual o estado que desejamos baixar e instanciamos a classe.\
É possível também ver o atributo `url` para entender qual a url que será utilizada para baixar o arquivo.


In [ ]:
estado = 'AC'

car = geo.providers.br.sfb.sicar.CAR(uf=estado)
car.url

<br>


[Only return the NumberOfFeatures in a WFS query](https://gis.stackexchange.com/questions/45101/nly-return-the-numberoffeatures-in-a-wfs-query) indica que o WFS na versão 1.0.0 não retorna.


In [ ]:
url = 'https://geoserver.car.gov.br/geoserver/sicar/ows?service=WFS&version=1.0.0&request=GetFeature&typeName=sicar%3Asicar_imoveis_sp&outputFormat=application%2Fjson'
url = 'https://geoserver.car.gov.br/geoserver/sicar/ows?service=WFS&version=1.0.0&request=GetFeature&typeName=sicar%3Asicar_imoveis_sp&resultType=hits'
url = 'https://geoserver.car.gov.br/geoserver/sicar/ows?service=WFS&version=1.0.0&request=GetFeature&typeName=sicar%3Asicar_imoveis_sp&outputFormat=application%2Fjson&startIndex=1000'
url

In [ ]:
car.create_session()
# r = car.session.get(url=url)
# print(r.status_code)

In [ ]:
xmin, ymin, xmax, ymax = map(float, gdf.total_bounds)
f'{xmin},{ymin},{xmax},{ymax}:{gdf.crs}'

<br>

O GeoServer versão 1.0.0, utiulizado pelo SFB, tem diversas limitações por ser desatualizado.

1. Não aceita o parâmetro `resultType=hits` para contar o número de feições existentes
2. O `maxFeatures` será, sempre, no máximo, de 10.000, mesmo que se defina mais que isso.


In [ ]:
base_url = "https://geoserver.car.gov.br/geoserver/sicar/ows"

params = {
    "service": "WFS",
    "version": "1.0.0",
    "request": "GetFeature",
    "typeName": "sicar:sicar_imoveis_ac",
    "outputFormat": "application/json",
    "maxFeatures": 10_000,
    #'BBOX': f'{xmin},{ymin},{xmax},{ymax}:{gdf.crs}',
    #'bbox': f'{xmin},{ymin},{xmax},{ymax}',
    "startIndex": 0,
    #'srsName': 'EPSG:4326',
    'srsName': 'EPSG:4674',
}
r = car.session.get(base_url, params=params)
print(len(r.json().get("features", [])))

In [ ]:
split_url = urlsplit(url=base_url)

In [ ]:
encode_params = urlencode(query=params, doseq=True)

In [ ]:
urlunsplit(
    (
        split_url.scheme,  # https
        split_url.netloc,  # servidor.com
        split_url.path,  # /wfs
        encode_params,  # Parâmetros modificados (a nova query)
        split_url.fragment,  # fragmento (#anchor)
    )
)

Na real


In [ ]:
params = {
    "service": "WFS",
    "version": "1.0.0",
    "request": "GetFeature",
    "typeName": "sicar:sicar_imoveis_sp",
    "outputFormat": "application/json",
    "maxFeatures": 10_000,
    #'BBOX': f'{xmin},{ymin},{xmax},{ymax}',
    #'srsName': 'EPSG:4326',
    'srsName': 'EPSG:4674',
}

total = 0
start = 0
all_features = []

while True:
    params["startIndex"] = start  # Pode não ser suportado, mas tente
    r = car.session.get(url=base_url, params=params, stream=True)
    r.raise_for_status()
    data = r.json()

    # Obtem os features
    features = data.get("features", [])
    count = len(features)
    print(f"Chunk com {count} feições")
    total += count

    #
    all_features.extend(features)
    print(params["maxFeatures"])
    if count < params["maxFeatures"]:
        break
    start += count

print(f"Total de feições: {total}")

In [ ]:
temp_path

In [ ]:
output_file = temp_path / f'car_{estado}4674.geojson'

In [ ]:
geojson = {"type": "FeatureCollection", "features": all_features}
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(geojson, f)

In [ ]:
gdf = gpd.read_file(output_file)
gdf.to_file(temp_path / f'car_{estado}4674.gpkg', driver='GPKG')

<br>

Fazemos o _download_ do arquivo.


In [ ]:
output_file = temp_path / f'car_{estado}.geojson'
car.download_file(filepath=output_file, url=url)

<br>

Por fim lemos o arquivo.


In [ ]:
gdf = gpd.read_file(filename=output_file)

gdf.info()
gdf.head(3)